# GenMoji — Colab LoRA training

Select **Runtime → Change runtime type → T4 GPU**, then run the setup cells in order. Opening this notebook from GitHub does not clone the repository. The repository must contain `pyproject.toml` and `src/genmoji/train_lora.py` on the selected branch.

**After a runtime disconnect:** reconnect a GPU, rerun **cells 1–6**, then run **cell 8 (resume)**. Skip cell 7 (new pilot). Cell 5 restores the exact training dataset from Drive so its hashes match the original run. Checkpoints and the standalone adapter are saved to Drive. Colab can disconnect even during a checkpoint write; choose an earlier intact checkpoint if the latest is incomplete.


## 1. Clone the project and check GPU

This assumes the GitHub repository is public. If you already cloned the branch, it is reused without changing it. A runtime reset erases `/content`.


In [ ]:
import os, sys
from pathlib import Path

root = Path('/content/GenMoji')
if not root.exists():
    !git clone --branch diffusion https://github.com/BettiM7/GenMoji.git /content/GenMoji
assert root.exists(), 'Clone failed; check the output above.'
os.chdir(root)
print('Working directory:', Path.cwd())
print('Python:', sys.version)
assert sys.version_info >= (3, 12), 'Python 3.12 or later is required.'
assert (root / 'pyproject.toml').is_file() and (root / 'src/genmoji/train_lora.py').is_file(), 'Push the packaging and training code to the diffusion branch first.'
!nvidia-smi


## 2. Install the project dependencies

Run again after a runtime reset. Installation and the first model download require internet access.


In [ ]:
%pip install -e ".[data,diffusion]"
import torch
assert torch.cuda.is_available(), 'Choose a GPU runtime and reconnect.'
print('GPU:', torch.cuda.get_device_name(0))


## 3. Mount Google Drive

Authorize access when prompted. The output folder is created when training starts. Checkpoints can take several GB; make sure Drive has enough free space.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

drive_root = Path('/content/drive/MyDrive/GenMoji')
drive_root.mkdir(parents=True, exist_ok=True)
output = drive_root / 'lora'
data_archive = drive_root / 'openmoji-512.tar.gz'


## 4. Download OpenMoji if needed

If the dataset was previously saved to Drive, cell 5 restores it and this cell skips the source download. If an OpenMoji checkout already exists at `/content/data/raw/openmoji`, it is reused.


In [ ]:
import shutil
source = root / 'data/raw/openmoji'
existing = Path('/content/data/raw/openmoji')
if not data_archive.exists() and not source.exists():
    source.parent.mkdir(parents=True, exist_ok=True)
    if existing.exists():
        shutil.move(str(existing), str(source))
    else:
        !git clone --depth 1 --filter=blob:none --sparse --branch 17.0.0 https://github.com/hfg-gmuend/openmoji.git data/raw/openmoji
        assert source.exists(), 'OpenMoji clone failed; check the output above.'
        !git -C data/raw/openmoji sparse-checkout set color/svg data
        assert (source / 'data/openmoji.json').is_file(), 'OpenMoji checkout failed; check the output above.'
print('Dataset backup on Drive:', data_archive.exists())


## 5. Prepare or restore the dataset

The first run renders the 512px dataset, then saves an archive to Drive. On reconnect it restores that exact archive instead of rerendering. If a prepared dataset already exists, it is kept. The archive contains images, captions, provenance and attribution.


In [ ]:
import tarfile
prepared = root / 'data/processed/openmoji-512'
if not prepared.exists():
    if data_archive.exists():
        prepared.parent.mkdir(parents=True, exist_ok=True)
        with tarfile.open(data_archive, 'r:gz') as archive:
            archive.extractall(prepared.parent, filter='data')
        print('Restored dataset from Drive')
    else:
        !python -m genmoji.cli prepare
        assert prepared.exists(), 'Dataset preparation failed; check the output above.'

!python scripts/check_dataset.py "{prepared}"
assert get_ipython().user_ns.get("_exit_code") == 0, "Dataset validation failed; check the output above."
if not data_archive.exists():
    temporary = data_archive.with_suffix('.tmp')
    with tarfile.open(temporary, 'w:gz') as archive:
        archive.add(prepared, arcname='openmoji-512')
    temporary.replace(data_archive)
    print('Saved dataset backup:', data_archive)
else:
    print('Using existing dataset backup:', data_archive)


## 6. Download the pinned Diffusers training script

Rerun after a runtime reset. The launcher checks that this is the unmodified v0.35.1 trainer.


In [ ]:
trainer = root / 'external/diffusers'
trainer.parent.mkdir(parents=True, exist_ok=True)
if not trainer.exists():
    !git clone --depth 1 --branch v0.35.1 https://github.com/huggingface/diffusers.git external/diffusers
assert (trainer / 'examples/text_to_image/train_text_to_image_lora.py').is_file(), 'Trainer clone failed; check the output above.'
print('Trainer:', trainer)


## 7. Start a new 50-step pilot

Run this **once for a new output folder**. Training uses a T4 GPU, FP16, rank 4 and gradient checkpointing. It downloads SD 1.5 the first time and shows progress and an ETA. Checkpoints are saved at steps 25 and 50. If the output already contains a run, skip this cell and use the resume cell.


In [ ]:
!python -m genmoji.cli diffusion train --device cuda --steps 50 --checkpoint-every 25 --keep-checkpoints 2 --output "{output}"


## 8. Resume or extend training

Run after the pilot, or after reconnecting and rerunning cells 1–6. `--steps 1000` is the total target, including completed updates. `latest` chooses the highest-numbered checkpoint. If it was interrupted during saving, inspect the folders below and replace `latest` with an earlier intact name such as `checkpoint-250`. The launcher needs `training.json`, the checkpoint state and the same dataset and training settings. If no checkpoint was saved yet, use a **new output folder** and restart the pilot.


In [ ]:
print('Available checkpoints:')
for path in sorted(output.glob('checkpoint-*')):
    print(path.name, '(optimizer present:', (path / 'optimizer.bin').is_file(), ')')


In [ ]:
!python -m genmoji.cli diffusion train --device cuda --resume latest --steps 1000 --checkpoint-every 250 --keep-checkpoints 2 --output "{output}"


## 9. Find the adapter

After training completes, `pytorch_lora_weights.safetensors` and run metadata are in `My Drive/GenMoji/lora/`. Keep the checkpoint folders if you may want to resume; the adapter alone cannot resume optimizer state.

If the Drive dataset archive or checkpoint is missing or corrupt, restore it from a backup or start a new run. Do not overwrite an existing run with a regenerated dataset. GPU access, runtime duration and available storage depend on Colab and your account.


## Updating from GitHub

Push the notebook and code changes from your computer to the `diffusion` branch first. In Colab, run `%cd /content/GenMoji` (with a single `%`) followed by `!git status` and `!git pull --ff-only origin diffusion`. If Git reports local changes or a conflict, inspect them before proceeding; do not reset them blindly. A pull updates the file on disk, not the notebook currently open in the browser: reopen this notebook from GitHub or the runtime file to see updated cells. Rerun dependency installation if dependencies changed. Resume training only with a compatible trainer and the same saved dataset and settings.
